# Paper Figure Reproduction Map

This notebook is one chapter of the runnable `KnottedGraph` user guide.  It is
generated into `User_guide/09_paper_figure_map.ipynb` so users can open the specific workflow
they need without navigating one very large notebook.

- self-contained after the shared setup cells


## 0. Setup, Preflight, And Shared Plot Style

The whole notebook uses the same visual convention:

- blue: surfaces, skeleton points, and graph edges;
- red: graph vertices;
- black axes;
- paper notation: `Upsilon(G; Y)`.

The helper functions in this section remove repeated plotting boilerplate from
the rest of the notebook.  This is also the library-level pattern worth
promoting later into public visualization helpers.


In [1]:
from pathlib import Path
import sys
import importlib.util
import os
import tempfile

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

DOC_ROOT = PROJECT_ROOT / "doc"
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "knottedgraph-mpl"))

print("project paths configured")
for package in ["numpy", "networkx", "sympy", "plotly", "matplotlib", "pyvista"]:
    print(f"{package:10s} = {importlib.util.find_spec(package) is not None}")


project paths configured
numpy      = True
networkx   = True
sympy      = True
plotly     = True
matplotlib = True
pyvista    = True


In [2]:
import math
import time

import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
import sympy as sp
from IPython.display import Math, display

from knotted_graph.projection import (
    compute_yamada_polynomial,
    sample_projections,
    select_projection,
)
from knotted_graph.visualization import plot_3D_graph_plotly

BLUE = "#1f77b4"
RED = "#d62728"
CAMERA = dict(eye=dict(x=1.45, y=1.55, z=1.18))
pio.renderers.default = "notebook_connected"
Y = sp.Symbol("Y")
kx, ky, kz = sp.symbols("k_x k_y k_z", real=True)


def axis_style():
    return dict(
        visible=True,
        title="",
        showticklabels=False,
        showbackground=False,
        showgrid=False,
        zeroline=False,
        showline=True,
        linecolor="black",
        linewidth=2,
    )


def apply_kg_layout(fig, *, width=760, height=620):
    fig.update_layout(
        title=None,
        width=width,
        height=height,
        margin=dict(l=0, r=0, t=0, b=0),
        scene=dict(
            xaxis=axis_style(),
            yaxis=axis_style(),
            zaxis=axis_style(),
            aspectmode="data",
            camera=CAMERA,
        ),
    )
    return fig


def plot_surface_polydata(surface, *, opacity=0.58):
    mesh = surface.triangulate()
    faces = mesh.faces.reshape(-1, 4)[:, 1:]
    pts = mesh.points
    fig = go.Figure(
        go.Mesh3d(
            x=pts[:, 0],
            y=pts[:, 1],
            z=pts[:, 2],
            i=faces[:, 0],
            j=faces[:, 1],
            k=faces[:, 2],
            color=BLUE,
            opacity=opacity,
        )
    )
    return apply_kg_layout(fig)


def plot_points_3d(points, *, size=3):
    points = np.asarray(points)
    fig = go.Figure(
        go.Scatter3d(
            x=points[:, 0],
            y=points[:, 1],
            z=points[:, 2],
            mode="markers",
            marker=dict(size=size, color=BLUE),
        )
    )
    return apply_kg_layout(fig)


def plot_graph_kg(graph):
    return apply_kg_layout(plot_3D_graph_plotly(graph))


def print_upsilon(label, expr):
    print(f"Upsilon({label}; Y) = {sp.expand(expr)}")


def display_bloch_vector(label, components):
    display(Math(label + r"=" + sp.latex(sp.Matrix(components))))


print("shared plotting and notation helpers ready")


shared plotting and notation helpers ready


In [3]:
from knotted_graph.applications.nodal import NodalSkeleton
from knotted_graph.applications.nodal.models import (
    awesome_bloch_vector,
    hopf_link_bloch_vector,
    pq_torus_knot_bloch_vector,
    solomon_bloch_vector,
    threelink_bloch_vector,
    trefoil_bloch_vector,
    unknot_bloch_vector,
)

print("nodal application imports ready")


nodal application imports ready


## 9. Paper Figure Reproduction Map

This section regenerates representative manuscript-style panels from public
notebook code.  Each plotted output below is generated in the cell that
immediately precedes it.


In [4]:
from knotted_graph.applications.nodal.models import pq_torus_knot_bloch_vector, threelink_bloch_vector

torus_ske = NodalSkeleton(
    pq_torus_knot_bloch_vector(1, 2, 0.2, k_symbols=(kx, ky, kz)),
    k_symbols=(kx, ky, kz),
    dimension=96,
    axis_scale=(1.0, 1.0, 1.5),
)
torus_surface = torus_ske.exceptional_surface_pv.connectivity("largest")
torus_points = torus_ske.skeleton_coords
torus_graph = torus_ske.skeleton_graph(simplify=True, smooth_epsilon=2)

print("torus surface =", (torus_surface.n_points, torus_surface.n_cells))
print("torus skeleton points =", torus_points.shape)
print("torus graph =", (torus_graph.number_of_nodes(), torus_graph.number_of_edges()))


torus surface = (8220, 16440)
torus skeleton points = (258, 3)
torus graph = (1, 1)


In [5]:
fig = plot_surface_polydata(torus_surface, opacity=0.58)
fig.show()


In [6]:
fig = plot_points_3d(torus_points, size=3)
fig.show()


In [7]:
fig = plot_graph_kg(torus_graph)
fig.show()


In [8]:
planarity_ske = NodalSkeleton(
    threelink_bloch_vector(0.41, k_symbols=(kx, ky, kz)),
    k_symbols=(kx, ky, kz),
    dimension=96,
    axis_scale=(1.0, 1.0, 1.5),
)
planarity_surface = planarity_ske.exceptional_surface_pv.connectivity("largest")
planarity_graph = planarity_ske.skeleton_graph(simplify=True, smooth_epsilon=2)
is_planar = nx.check_planarity(nx.Graph(planarity_graph))[0]

print("three-link surface =", (planarity_surface.n_points, planarity_surface.n_cells))
print("three-link graph =", (planarity_graph.number_of_nodes(), planarity_graph.number_of_edges()))
print("is_planar =", is_planar)


three-link surface = (14192, 28396)
three-link graph = (6, 9)
is_planar = False


In [9]:
fig = plot_surface_polydata(planarity_surface, opacity=0.58)
fig.show()


In [10]:
fig = plot_graph_kg(planarity_graph)
fig.show()
